In [1]:
import torch, os
import numpy as np
import fig5
import matplotlib.pyplot as plt
from fig5 import zscore
device=torch.device("cuda")

In [2]:
dt = 2
config = {'gains':  [1, 1, 1.35], 
          'nonsym' : [0, 1, 1], 
          'relu' : [0, 0, 1], 
          'distribution' : ['uniform', 'uniform', 'sparse'],
          'names': ['symm', 'non-symm', 'echo-state network']}

# the simulations take a while even on a good GPU, so we save the results to disk
local_path = '/media/carsen/disk1/random_matrix'


In [3]:
nstim = 50
tisi = 3000//dt
wm_isi = np.linspace(60, tisi,51)[:-1].astype('int32')
acc, acc_shuff = fig5.run_multishot(config, nstim, tisi, wm_isi, niter = 20)

np.savez(os.path.join(local_path, 'acc_multi.npz') , acc = acc, wm_isi = wm_isi, acc_shuff = acc_shuff)

nstim = 50
tisi = 3000//dt
wm_isi = np.linspace(60, tisi,51)[:-1].astype('int32')
acc, acc_shuff = fig5.run_zeroshot(config, nstim, tisi, wm_isi, niter = 20)

np.savez(os.path.join(local_path, 'acc_zero.npz'), acc = acc, wm_isi = wm_isi, acc_shuff = acc_shuff)

nstim = 15
tisi = 9000//dt
wm_isi = np.linspace(60, tisi,51)[:-1].astype('int32')
acc, acc_shuff = fig5.run_zeroshot(config, nstim, tisi, wm_isi, niter = 20, mode = 'aligned')
np.savez(os.path.join(local_path, 'acc_zero_aligned.npz'), acc = acc, wm_isi = wm_isi, acc_shuff = acc_shuff)

In [ ]:
data1 = np.load(os.path.join(local_path, 'acc_multi2.npz'),allow_pickle = True)
data2 = np.load(os.path.join(local_path, 'acc_zero2.npz'),allow_pickle = True)
data3 = np.load(os.path.join(local_path, 'acc_zero_aligned2.npz'),allow_pickle = True)


In [ ]:
# example raster periods are saved directly to disk
dat1 = np.load('fig5_raster1.npz', allow_pickle = True)
dat2 = np.load('fig5_raster2.npz', allow_pickle = True)

In [ ]:
# pngs for schematics
im = plt.imread('fig5_1.png')
im1 = plt.imread('fig5_2.png')
im2 = plt.imread('fig5_3.png')
im3 = plt.imread('fig5_4.png')


In [ ]:
import matplotlib
import string
import matplotlib.transforms as mtransforms
from matplotlib import rcParams

default_font = 12
rcParams["font.family"] = "Arial"
rcParams["savefig.dpi"] = 300
rcParams["axes.spines.top"] = False
rcParams["axes.spines.right"] = False
rcParams["axes.titlelocation"] = "left"
rcParams["axes.titleweight"] = "normal"
rcParams["font.size"] = default_font
ltr = string.ascii_lowercase
def plot_label(ltr, il, ax, trans, fs_title=20):
    ax.text(
        0.0,
        1.0,
        ltr[il],
        transform=ax.transAxes + trans,
        va="bottom",
        fontsize=fs_title,
        fontweight="bold",
    )
    il += 1
    return il
il = 0
cmap = plt.get_cmap("Dark2")

fig = plt.figure(figsize=(14,6))

grid = plt.GridSpec(5,6, figure=fig, left=0.02, right=0.98, top=0.98, bottom=0.02, wspace = 0.35, hspace = 0.5)

ax = plt.subplot(grid[:2,:2])
plt.imshow(im)
plt.axis('off')
plt.title('persistent activity model', loc = 'center')
transl = mtransforms.ScaledTranslation(-20 / 72, 0 / 72, fig.dpi_scale_trans)
il = plot_label(ltr, il, ax, transl)

grid2 = matplotlib.gridspec.GridSpecFromSubplotSpec(6,2, subplot_spec=grid[:2,2:], wspace=0.1, hspace=0.2)

ax = plt.subplot(grid2[:5,0])
plt.imshow(zscore(dat1['Xsort'], axis=-1), vmax = .5, vmin = 0, aspect = 'auto', cmap = 'grey_r')
plt.xlim([0, dat1['Xsort'].shape[-1]])
plt.axis('off')
plt.title('symmetric dynamics', color = cmap(0))
transl = mtransforms.ScaledTranslation(-20 / 72, 0 / 72, fig.dpi_scale_trans)
il = plot_label(ltr, il, ax, transl)

ax = plt.subplot(grid2[5,0])
plt.plot(dat1['xx'], lw = 1, color = 'k')
plt.xlim([0, len(dat1['xx'])])
plt.axis('off')
plt.text(-.02, .5, 'inputs', ha = 'right', va = 'center', transform= ax.transAxes)


ax = plt.subplot(grid2[:5,1])
plt.imshow(zscore(dat2['Xsort'], axis=-1), vmax = .5, vmin = 0, aspect = 'auto', cmap = 'grey_r')
plt.xlim([0, dat2['Xsort'].shape[-1]])
plt.axis('off')
plt.title('non-symmetric dynamics', color = cmap(1))
transl = mtransforms.ScaledTranslation(-20 / 72, 0 / 72, fig.dpi_scale_trans)
il = plot_label(ltr, il, ax, transl)

ax = plt.subplot(grid2[5,1])
plt.plot(dat2['xx'], lw = 1, color = 'k')
plt.xlim([0, len(dat2['xx'])])
plt.axis('off');

#grid2 = matplotlib.gridspec.GridSpecFromSubplotSpec(3,2, subplot_spec=grid[2:,:2], wspace=0.3, hspace=0.3)

ax = plt.subplot(grid[2,:2])
plt.imshow(im1)#, aspect = 'auto')
plt.title('trained working memory (binary)')
plt.axis('off')
transl = mtransforms.ScaledTranslation(-20 / 72, 0 / 72, fig.dpi_scale_trans)
il = plot_label(ltr, il, ax, transl)

fig5.plot_with_time(cmap, config, data1, grid, k = 0, label = True)

#grid3 = matplotlib.gridspec.GridSpecFromSubplotSpec(3,2, subplot_spec=grid[2:,2:4], wspace=0.3, hspace=0.3)
ax = plt.subplot(grid[2,2:4])
plt.imshow(im2)#, aspect = 'auto')
plt.title('zero-shot working memory')
plt.axis('off')
transl = mtransforms.ScaledTranslation(-20 / 72, 0 / 72, fig.dpi_scale_trans)
il = plot_label(ltr, il, ax, transl)

fig5.plot_with_time(cmap, config, data2, grid, k = 2)

#grid4 = matplotlib.gridspec.GridSpecFromSubplotSpec(3,2, subplot_spec=grid[2:,4:6], wspace=0.3, hspace=0.3)
ax = plt.subplot(grid[2,4:6])
plt.imshow(im3)#, aspect = 'auto')
plt.title('zero-shot w/ subspace-aligned inputs')
plt.axis('off')
transl = mtransforms.ScaledTranslation(-20 / 72, 0 / 72, fig.dpi_scale_trans)
il = plot_label(ltr, il, ax, transl)

fig5.plot_with_time(cmap, config, data3, grid, k = 4)

#fig.savefig("/mnt/disk1/code/random_matrix/fig5.pdf", dpi=300, bbox_inches = 'tight')